In [1]:
import pandas as pd
df = pd.read_csv(r"E:\Sindhura\DataSets\car_price_prediction.csv",na_values=["-","NA",""],keep_default_na=False)
df.head()

,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
0,45654403,13328,1399.0,LEXUS,RX 450,2010,Jeep,Yes,Hybrid,3.5,186005 km,6.0,Automatic,4x4,04-May,Left wheel,Silver,12
1,44731507,16621,1018.0,CHEVROLET,Equinox,2011,Jeep,No,Petrol,3,192000 km,6.0,Tiptronic,4x4,04-May,Left wheel,Black,8
2,45774419,8467,NaN,HONDA,FIT,2006,Hatchback,No,Petrol,1.3,200000 km,4.0,Variator,Front,04-May,Right-hand drive,Black,2
3,45769185,3607,862.0,FORD,Escape,2011,Jeep,Yes,Hybrid,2.5,168966 km,4.0,Automatic,4x4,04-May,Left wheel,White,0
4,45809263,11726,446.0,HONDA,FIT,2014,Hatchback,Yes,Petrol,1.3,91901 km,4.0,Automatic,Front,04-May,Left wheel,Silver,4


In [2]:
len(df)

19237

In [3]:
df.shape

(19237, 18)

In [4]:
def fetch_km(data):
    x=data.split()
    return x[0]

In [5]:
df['Mileage']=df['Mileage'].apply(fetch_km).astype(int)
df['Mileage']

0        186005
1        192000
2        200000
3        168966
4         91901
          ...  
19232    300000
19233    161600
19234    116365
19235     51258
19236    186923
Name: Mileage, Length: 19237, dtype: int64

In [6]:
df.dtypes

ID                    int64
Price                 int64
Levy                float64
Manufacturer         object
Model                object
Prod. year            int64
Category             object
Leather interior     object
Fuel type            object
Engine volume        object
Mileage               int64
Cylinders           float64
Gear box type        object
Drive wheels         object
Doors                object
Wheel                object
Color                object
Airbags               int64
dtype: object

In [7]:
df['Levy'] = df['Levy'].fillna(0).astype(int)

## Outlier handling

In [8]:
cat = list(df.select_dtypes(include='object').columns) 
con = list(df.select_dtypes(include='number').columns)

In [9]:
import numpy as np
out_summary = {}
for col in con:

    q1 = np.percentile(df[col], 25)
    q3 = np.percentile(df[col], 75)

    iqr = q3 - q1 

    minimun = q1 - 1.5 * iqr
    maximum = q3 + 1.5 * iqr

    lower_out = df[col] < minimun
    upper_out = df[col] > maximum

    outliers = df[lower_out | upper_out]

    print("=================")
    print(f"Outliers less than minimum: {df[lower_out]}")
    print(f"Outliers more than maximum: {df[upper_out]}")
    print(f"number of outliers for {col}: {len(outliers)}")
    out_summary[col] = {
        "lower": minimun,
        "upper": maximum,
        "outlier_count": len(outliers),
        "outlier_pct": round(len(outliers)/len(df) * 100, 2)
    }

out_report = pd.DataFrame(out_summary).T
out_report

Outliers less than minimum:              ID  Price  Levy   Manufacturer           Model  Prod. year  \
1      44731507  16621  1018      CHEVROLET         Equinox        2011   
15     45465200    549  1079         TOYOTA             CHR        2018   
21     45168471  20698   586         TOYOTA           Prius        2012   
30     44944581  15681  1288  MERCEDES-BENZ            Vito        2007   
34     45542380  24462     0           JEEP  Grand Cherokee        2007   
...         ...    ...   ...            ...             ...         ...   
19176  45511664  14113   531        HYUNDAI        Veloster        2012   
19192  45172859  14113     0          SCION              xD        2010   
19208  45433278  34811   779         TOYOTA           RAV 4        2013   
19223  44571028   8781     0          DODGE         Caliber        2008   
19229  39977395     50     0         TOYOTA           Prius        2008   

          Category Leather interior       Fuel type Engine volume  Mile

,lower,upper,outlier_count,outlier_pct
ID,45542881.0,45957529.0,2531.0,13.16
Price,-19785.0,47191.0,1073.0,5.58
Levy,-1375.5,2292.5,161.0,0.84
Prod. year,2000.0,2024.0,982.0,5.10
Mileage,-107984.5,367011.5,640.0,3.33
Cylinders,4.0,4.0,4870.0,25.32
Airbags,-8.0,24.0,0.0,0.00


In [10]:
def handle_outlier(df,col,outlier_percent,mini,maxi):
    lower_index = list(df[df[col]<mini].index) # contains index of all such values which are below min
    upper_index = list(df[df[col]>maxi].index) # contains index of all such values which are above max
    all_ = lower_index + upper_index
    # Perform outlier handling on basis of percentage of outliers present in the data
    df_new = df.copy()
    if outlier_percent<10:
        df_new = df.drop(index=all_) # drops on basis of index we have provided
    else:
        df_new[col] = df[col].clip(mini,maxi)
    return df_new

In [11]:
df2 = df.copy()
for i in con:
    # df.loc[row_name,column_name]
    out_perc = out_report.loc[i,'outlier_pct']
    mini = out_report.loc[i,'lower']
    maxi = out_report.loc[i,'upper']
    df2 = handle_outlier(df2,i,out_perc,mini,maxi=maxi) # positional arguments,keyword arguments
df2

,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
0,45654403,13328,1399,LEXUS,RX 450,2010,Jeep,Yes,Hybrid,3.5,186005,4.0,Automatic,4x4,04-May,Left wheel,Silver,12
1,45542881,16621,1018,CHEVROLET,Equinox,2011,Jeep,No,Petrol,3,192000,4.0,Tiptronic,4x4,04-May,Left wheel,Black,8
2,45774419,8467,0,HONDA,FIT,2006,Hatchback,No,Petrol,1.3,200000,4.0,Variator,Front,04-May,Right-hand drive,Black,2
3,45769185,3607,862,FORD,Escape,2011,Jeep,Yes,Hybrid,2.5,168966,4.0,Automatic,4x4,04-May,Left wheel,White,0
4,45809263,11726,446,HONDA,FIT,2014,Hatchback,Yes,Petrol,1.3,91901,4.0,Automatic,Front,04-May,Left wheel,Silver,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19231,45772306,5802,1055,MERCEDES-BENZ,E 350,2013,Sedan,Yes,Diesel,3.5,107800,4.0,Automatic,Rear,04-May,Left wheel,Grey,12
19233,45778856,15681,831,HYUNDAI,Sonata,2011,Sedan,Yes,Petrol,2.4,161600,4.0,Tiptronic,Front,04-May,Left wheel,Red,8
19234,45804997,26108,836,HYUNDAI,Tucson,2010,Jeep,Yes,Diesel,2,116365,4.0,Automatic,Front,04-May,Left wheel,Grey,4
19235,45793526,5331,1288,CHEVROLET,Captiva,2007,Jeep,Yes,Diesel,2,51258,4.0,Automatic,Front,04-May,Left wheel,Black,4


## separate x and y

In [12]:
X = df2.drop(columns=['ID',"Price","Gear box type","Drive wheels","Doors","Wheel"])
Y = df2[['Price']]

In [13]:
for i in ['Manufacturer','Model','Category','Leather interior','Fuel type','Engine volume','Color']:
    print(i,X[i].unique(),sep='\n')

Manufacturer
['LEXUS' 'CHEVROLET' 'HONDA' 'FORD' 'HYUNDAI' 'TOYOTA' 'MERCEDES-BENZ'
 'PORSCHE' 'BMW' 'JEEP' 'VOLKSWAGEN' 'AUDI' 'RENAULT' 'NISSAN' 'SUBARU'
 'DAEWOO' 'KIA' 'MITSUBISHI' 'SSANGYONG' 'MAZDA' 'GMC' 'FIAT' 'INFINITI'
 'ALFA ROMEO' 'SUZUKI' 'OPEL' 'ACURA' 'CITROEN' 'MINI' 'DODGE' 'CHRYSLER'
 'ISUZU' 'VAZ' 'JAGUAR' 'SKODA' 'LINCOLN' 'BUICK' 'DAIHATSU' 'CADILLAC'
 'PEUGEOT' 'VOLVO' 'სხვა' 'HAVAL' 'LAND ROVER' 'SCION' 'MERCURY' 'SEAT'
 'LANCIA' 'MASERATI' 'HUMMER' 'SAAB' 'GAZ' 'UAZ' 'PONTIAC' 'SATURN'
 'GREATWALL']
Model
['RX 450' 'Equinox' 'FIT' ... '406' 'Prius C aqua' 'Optima ex']
Category
['Jeep' 'Hatchback' 'Sedan' 'Microbus' 'Goods wagon' 'Universal' 'Coupe'
 'Minivan' 'Cabriolet' 'Pickup' 'Limousine']
Leather interior
['Yes' 'No']
Fuel type
['Hybrid' 'Petrol' 'Diesel' 'Plug-in Hybrid' 'CNG' 'LPG' 'Hydrogen']
Engine volume
['3.5' '3' '1.3' '2.5' '2' '1.8' '2.4' '1.6' '2.0 Turbo' '2.2 Turbo' '1.5'
 '4.4' '3.3' '3.0 Turbo' '1.4 Turbo' '3.6' '4' '2.3' '1.5 Turbo'
 '1.6 Turbo

**Manufacturer and Engine volume has too many unique features. will apply Label encoding to it**

In [14]:
cat = list(X.select_dtypes(include='object').columns) 
con = list(X.select_dtypes(include='number').columns)
print(cat)
print(con)

['Manufacturer', 'Model', 'Category', 'Leather interior', 'Fuel type', 'Engine volume', 'Color']
['Levy', 'Prod. year', 'Mileage', 'Cylinders', 'Airbags']


In [15]:
cat_onehot = ['Model', 'Category', 'Leather interior', 'Fuel type', 'Color']
cat_label = ['Manufacturer','Engine volume']

In [16]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.compose import ColumnTransformer

num_pipe = make_pipeline(SimpleImputer(strategy='median'),
                         StandardScaler())

cat_pipe = make_pipeline(SimpleImputer(strategy='most_frequent'),
                         OneHotEncoder(handle_unknown='ignore',sparse_output=False))

pre = ColumnTransformer([('cat',cat_pipe,cat_onehot),
                         ('con',num_pipe,con)]).set_output(transform='pandas')

from sklearn.model_selection import train_test_split
xtrain,xtest,ytrain,ytest = train_test_split(X,Y,random_state=21,train_size=0.75)

pre.fit(xtrain)

xtrain_pre = pre.transform(xtrain)
xtest_pre = pre.transform(xtest)

# Apply LabelEncoder separately
encoders = {}

for col in cat_label:
    # Impute missing values using training data's most frequent value
    most_frequent = xtrain[col].mode().iloc[0]

    train_col = xtrain[col].fillna(most_frequent).astype(str)
    test_col = xtest[col].fillna(most_frequent).astype(str)

    le = LabelEncoder()
    xtrain_pre[col] = le.fit_transform(train_col)

    # Handle categories in test data that weren't present in training
    mapping = {v: i for i, v in enumerate(le.classes_)}
    xtest_pre[col] = test_col.map(mapping).fillna(-1).astype(int)

    encoders[col] = le

In [33]:
xtrain_pre.columns

Index(['cat__Model_09-Mar', 'cat__Model_100', 'cat__Model_100 NX',
       'cat__Model_1000', 'cat__Model_114', 'cat__Model_116', 'cat__Model_118',
       'cat__Model_128', 'cat__Model_128 M tech', 'cat__Model_130',
       ...
       'cat__Color_Sky blue', 'cat__Color_White', 'cat__Color_Yellow',
       'con__Levy', 'con__Prod. year', 'con__Mileage', 'con__Cylinders',
       'con__Airbags', 'Manufacturer', 'Engine volume'],
      dtype='object', length=1180)

## Algorithm Evaluation

In [17]:
import re

# Replace [, ], and < with an underscore
xtrain_pre.columns = [re.sub(r'[\[\]<]', '_', str(col)) for col in xtrain_pre.columns]
xtest_pre.columns = [re.sub(r'[\[\]<]', '_', str(col)) for col in xtest_pre.columns]


In [18]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor

In [19]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [20]:
from xgboost import XGBRegressor

In [21]:
dct = {
    "Linear":LinearRegression(),
    "Decision":DecisionTreeRegressor(),
    "Random":RandomForestRegressor(),
    "Gradient":GradientBoostingRegressor(),
    "XGB":XGBRegressor()
}

In [22]:
r2 = []
from sklearn.metrics import r2_score
for model in dct.values():
    model.fit(xtrain_pre,ytrain['Price'])
    preds = model.predict(xtest_pre)
    score = r2_score(ytest,preds)
    r2.append(score)

In [23]:
r2

[0.3897110904655585,
 0.600043384165748,
 0.7498815527322871,
 0.5782403136332779,
 0.6993575096130371]

In [24]:
model_results = pd.DataFrame(list(zip(list(dct.keys()),r2)),columns=["Model Name","R2 Scores"]).sort_values(by="R2 Scores",ascending=False)
model_results

,Model Name,R2 Scores
2,Random,0.749882
4,XGB,0.699358
1,Decision,0.600043
3,Gradient,0.578240
0,Linear,0.389711


In [29]:
params ={'n_estimators':[50,100,125,150],
        'max_depth':[10,15,20,25]}

In [30]:
from sklearn.model_selection import RandomizedSearchCV
rscv = RandomizedSearchCV(estimator=RandomForestRegressor(),param_distributions=params,cv=3,scoring='r2')
rscv.fit(xtrain_pre,ytrain['Price'])

RandomizedSearchCV(cv=3, estimator=RandomForestRegressor(),
                   param_distributions={'max_depth': [10, 15, 20, 25],
                                        'n_estimators': [50, 100, 125, 150]},
                   scoring='r2')

In [31]:
rscv.best_params_

{'n_estimators': 125, 'max_depth': 25}

In [32]:
rscv.best_score_

np.float64(0.705659361756362)